# Kubeflow - VertexAI pipelines tutorial
## Installing required libraries

In [ ]:
! pip install -U "google-auth>=2.30.0" "google-cloud-aiplatform>=1.60.0" "kfp>=2.7.0"

## Define your values

In [ ]:
import random
import string

PROJECT_ID = "edem-practica-mlops"
LOCATION = "us-central1"
random_suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=8))
BUCKET_NAME = f"{PROJECT_ID}-bucket-{random_suffix}"
PIPELINE_ROOT = f"gs://{BUCKET_NAME}/pipeline_root/"

BQ_LOCATION = LOCATION.split("-")[0].upper()
BUCKET_URI = "gs://"+BUCKET_NAME

In [ ]:
# Create a bucket:
!gsutil mb -l $LOCATION gs://$BUCKET_NAME

In [ ]:
# Service aacc
shell_output = !gcloud auth list 2>/dev/null
SERVICE_ACCOUNT = shell_output[2].replace("*", "").strip()
print(SERVICE_ACCOUNT)

In [ ]:
!gsutil iam ch serviceAccount:{SERVICE_ACCOUNT}:roles/storage.objectCreator $BUCKET_URI
!gsutil iam ch serviceAccount:{SERVICE_ACCOUNT}:roles/storage.objectViewer $BUCKET_URI
!gsutil iam ch serviceAccount:{SERVICE_ACCOUNT}:roles/storage.admin $BUCKET_URI

## Initialize Vertex AI pipelines

In [ ]:
import google.cloud.aiplatform as aiplatform
import kfp
from kfp import compiler, dsl
from kfp.dsl import Artifact, Dataset, Input, Metrics, Model, Output, component

In [ ]:
aiplatform.init(project=PROJECT_ID, staging_bucket=BUCKET_URI)

## Linear pipeline

In [ ]:
@component
def start_step() -> str:
    return 'hello world'

@component
def print_step(my_var: str) -> str:
    print(f'the data artifact is: {my_var}')
    return my_var

@component
def final_step(my_var: str):
    print(f'the data artifact is still: {my_var}')

@dsl.pipeline
def linear_pipeline():
    my_var = start_step()
    my_var_2 = print_step(my_var=my_var.output)
    final_step(my_var=my_var_2.output)


compiler.Compiler().compile(pipeline_func=linear_pipeline, package_path='linear_pipeline.yaml')

In [ ]:
job = aiplatform.PipelineJob(
    display_name="linear_pipeline",
    template_path="linear_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    location=LOCATION
)

job.run()

## Branch pipeline

In [ ]:
@component
def step_a() -> int:
    return 1

@component
def step_b() -> int:
    return 2

@component
def join_results(a: int, b: int):
    print(f'a is {a}')
    print(f'b is {b}')
    print(f'total is {a + b}')


@dsl.pipeline
def branch_pipeline():
    a_result = step_a()
    b_result = step_b()

    join_results(a=a_result.output, b=b_result.output)



compiler.Compiler().compile(pipeline_func=branch_pipeline, package_path='branch_pipeline.yaml')

job = aiplatform.PipelineJob(
    display_name="branch_pipeline",
    template_path="branch_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    location=LOCATION
)

job.run()

## Foreach pipeline

In [ ]:
from kfp import dsl
from kfp.dsl import component, PipelineTask
from kfp.compiler import Compiler

@component
def process_title(title: str) -> str:
    return f"{title} processed"

@component
def join_results(processed_titles: list):
    for result in processed_titles:
        print(result)

@dsl.pipeline
def foreach_pipeline():
    titles = ['Stranger Things', 'House of Cards', 'Narcos']

    processed_tasks: list[PipelineTask] = []

    for title in titles:
        task = process_title(title=title)
        processed_tasks.append(task)

    join_results(processed_titles=[t.output for t in processed_tasks])


compiler.Compiler().compile(pipeline_func=foreach_pipeline, package_path='foreach_pipeline.yaml')

In [ ]:
job = aiplatform.PipelineJob(
    display_name="foreach_pipeline",
    template_path="foreach_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    location=LOCATION
)

job.run()

## Input parameters

In [ ]:
from kfp import dsl
from kfp.dsl import component
from kfp.compiler import Compiler

@component
def print_alpha(alpha: float):
    print(f'alpha is {alpha}')

@dsl.pipeline
def parameter_pipeline(alpha: float = 0.01):
    print_alpha(alpha=alpha)


compiler.Compiler().compile(pipeline_func=parameter_pipeline, package_path='parameter_pipeline.yaml')

In [ ]:
job = aiplatform.PipelineJob(
    display_name="parameter_pipeline",
    template_path="parameter_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    parameter_values={'alpha': 0.199},  # your custom alpha value
    location=LOCATION
)

job.run()

## Dealing with artifacts (datasets, models, etc.)

In [ ]:
from kfp.dsl import component, Input, Output, Dataset

@component(packages_to_install=['pandas'])
def generate_data(data_out: Output[Dataset]):
    import pandas as pd
    df = pd.DataFrame({'col': [1, 2, 3]})
    
    # Ensure the output directory exists
    import os
    os.makedirs(data_out.path, exist_ok=True)
    
    df.to_csv(f"{data_out.path}/data.csv", index=False)

@component(packages_to_install=['pandas'])
def consume_data(data_in: Input[Dataset]):
    import pandas as pd
    df = pd.read_csv(f"{data_in.path}/data.csv")
    print(df)
    
@dsl.pipeline
def dataset_pipeline():
    data = generate_data()
    consume_data(data_in = data.outputs["data_out"])
    
compiler.Compiler().compile(pipeline_func=dataset_pipeline, package_path='dataset_pipeline.yaml')

In [ ]:
job = aiplatform.PipelineJob(
    display_name="dataset_pipeline",
    template_path="dataset_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    location=LOCATION
)

job.run()

# Exercise: build the training pipeline 

In [ ]:
from kfp import dsl
from kfp.dsl import component, Input, Output, Dataset, Model
from kfp.compiler import Compiler

@component(packages_to_install=['scikit-learn', 'numpy'])
def ingest_data(X_out: Output[Dataset], y_out: Output[Dataset]):
    from sklearn.datasets import load_iris
    import numpy as np
    import os

    iris = load_iris()

    os.makedirs(X_out.path, exist_ok=True)
    os.makedirs(y_out.path, exist_ok=True)

    np.save(f"{X_out.path}/X.npy", iris.data)
    np.save(f"{y_out.path}/y.npy", iris.target)

@component(packages_to_install=['scikit-learn', 'numpy'])
def split_data(
    X_in: Input[Dataset],
    y_in: Input[Dataset],
    X_train_out: Output[Dataset],
    X_test_out: Output[Dataset],
    y_train_out: Output[Dataset],
    y_test_out: Output[Dataset],
    test_size: float = 0.2
):
    from sklearn.model_selection import train_test_split
    import numpy as np
    import os

    X = np.load(f"{X_in.path}/X.npy")
    y = np.load(f"{y_in.path}/y.npy")

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size)

    os.makedirs(X_train_out.path, exist_ok=True)
    os.makedirs(X_test_out.path, exist_ok=True)
    os.makedirs(y_train_out.path, exist_ok=True)
    os.makedirs(y_test_out.path, exist_ok=True)

    np.save(f"{X_train_out.path}/X_train.npy", X_train)
    np.save(f"{X_test_out.path}/X_test.npy", X_test)
    np.save(f"{y_train_out.path}/y_train.npy", y_train)
    np.save(f"{y_test_out.path}/y_test.npy", y_test)

@component(packages_to_install=['scikit-learn', 'numpy'])
def train(
    X_train: Input[Dataset],
    y_train: Input[Dataset],
    model_out: Output[Model],
    max_depth: int,
    n_estimators: int,
    random_state: int
):
    from sklearn.ensemble import RandomForestClassifier
    import pickle, numpy as np, os

    X = np.load(f"{X_train.path}/X_train.npy")
    y = np.load(f"{y_train.path}/y_train.npy")

    clf = RandomForestClassifier(
        max_depth=max_depth, 
        n_estimators=n_estimators, 
        random_state=random_state
    )
    clf.fit(X, y)

    os.makedirs(model_out.path, exist_ok=True)

    with open(f"{model_out.path}/model.pkl", 'wb') as f:
        pickle.dump(clf, f)

@component(packages_to_install=['scikit-learn', 'numpy'])
def show_metrics(
    model: Input[Model], 
    X_test: Input[Dataset], 
    y_test: Input[Dataset]
):
    from sklearn.metrics import classification_report, confusion_matrix
    import pickle, numpy as np

    X = np.load(f"{X_test.path}/X_test.npy")
    y = np.load(f"{y_test.path}/y_test.npy")

    with open(f"{model.path}/model.pkl", 'rb') as f:
        clf = pickle.load(f)

    y_pred = clf.predict(X)
    print(confusion_matrix(y, y_pred))
    print(classification_report(y, y_pred))

@dsl.pipeline
def training_pipeline(max_depth: int = 2, n_estimators: int = 100, random_state: int = 0):
    ingest = ingest_data()

    split = split_data(
        X_in=ingest.outputs['X_out'],
        y_in=ingest.outputs['y_out'],
    )

    model = train(
        X_train=split.outputs['X_train_out'],
        y_train=split.outputs['y_train_out'],
        max_depth=max_depth,
        n_estimators=n_estimators,
        random_state=random_state
    )

    show_metrics(
        model=model.outputs['model_out'],
        X_test=split.outputs['X_test_out'],
        y_test=split.outputs['y_test_out']
    )

compiler.Compiler().compile(pipeline_func=training_pipeline, package_path='training_pipeline.yaml')


In [ ]:
job = aiplatform.PipelineJob(
    display_name="training_pipeline",
    template_path="training_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    location=LOCATION
)

job.run()